# PEMDT-Cirrhosis — Latent Severity Staging + Treatment Transition-Fusion

A **multi-state latent severity HMM** for cirrhosis, and the headline methodological
experiment: **are treatments better modelled as *observations* of the current state, or as
*drivers* of future state transitions?** This generalises the PEMDT transition-fusion idea
(diabetes/insulin, vasopressors) to a new disease with a richer set of interventions.

## What this is
- **Latent ordinal stages** (Stage 0–3), fit by EM/Baum–Welch on the cirrhosis timeline.
  States are *latent* (not threshold-defined), then **relabelled by posterior-weighted MELD**
  so Stage 0 = least severe. Validated against *external* anchors: MELD, mortality,
  decompensation. (We deliberately do **not** define stages by MELD bands — that would be
  circular.)
- **Observations:** Bilirubin, INR, Albumin, Creatinine, Sodium, MAP (the strongest,
  least-redundant signals from the earlier diagnostic; missing values handled natively).
- **Transition-fusion experiment:** three model variants —
  `baseline` (no treatment), `emission` (treatment generated by the latent state →
  *observation of severity*), `transition` (treatment conditions the transition matrix →
  *driver of dynamics*) — compared by a **fair one-step-ahead predictive likelihood of the
  next labs**, so all three are scored on the same target.

## Honest caveats (important)
- **Causal language does not apply.** Treatments here suffer confounding by indication
  (lactulose is given *because* of encephalopathy, etc.). A better-fitting `transition` model
  shows treatment carries information about *dynamics*, **not** that it *causes* transitions.
  This is an **associational** comparison; a causal claim needs g-methods (out of scope).
- `GCS_total`/`HRV_proxy` from the source timeline are excluded (former mis-constructed,
  latter near-constant). A proper GCS re-extraction is a future improvement.
- Presented for review, not clinical use.

**Inputs:** `cirrhosis_timeline.csv`, `cirrhosis_cohort.csv` (from `PEMDT_cirrhosis.ipynb`).
**Treatment step** additionally reads MIMIC-IV `hosp/prescriptions` + `icu/inputevents`.


## HMM library (missing-aware diagonal-Gaussian HMM + fusion modes)

In [1]:
"""Missing-aware diagonal-Gaussian HMM with optional treatment fusion.
Modes:
  baseline   : Gaussian emissions, homogeneous transitions
  emission   : Gaussian + Bernoulli(treatment) emissions, homogeneous transitions
  transition : Gaussian emissions, treatment-conditioned transitions (A_off / A_on)
"""
import numpy as np

VAR_FLOOR = 1e-3

def emission_logB(O, mu, var, u=None, p_treat=None):
    """O: T x D (NaN allowed). returns logB: T x K. Optional Bernoulli treatment emission."""
    T, D = O.shape; K = mu.shape[0]
    mask = ~np.isnan(O)                                  # T x D
    diff = O[:, None, :] - mu[None, :, :]                # T x K x D
    term = -0.5 * (np.log(2*np.pi*var)[None] + (diff**2)/var[None])
    term = np.where(mask[:, None, :], term, 0.0)
    logB = term.sum(axis=2)                              # T x K
    if u is not None and p_treat is not None:
        pt = np.clip(p_treat, 1e-4, 1-1e-4)              # K
        logB = logB + (u[:, None]*np.log(pt)[None] + (1-u[:, None])*np.log(1-pt)[None])
    return logB

def fwd_bwd(logB, pi, A_t):
    """A_t: (T-1) x K x K time-indexed transition matrices. Scaled forward-backward."""
    T, K = logB.shape
    m = logB.max(axis=1); B = np.exp(logB - m[:, None])
    alpha = np.zeros((T, K)); c = np.zeros(T)
    alpha[0] = pi * B[0]; c[0] = alpha[0].sum() + 1e-300; alpha[0] /= c[0]
    for t in range(1, T):
        alpha[t] = (alpha[t-1] @ A_t[t-1]) * B[t]
        c[t] = alpha[t].sum() + 1e-300; alpha[t] /= c[t]
    beta = np.zeros((T, K)); beta[-1] = 1.0
    for t in range(T-2, -1, -1):
        beta[t] = (A_t[t] @ (B[t+1]*beta[t+1])) / c[t+1]
    gamma = alpha*beta; gamma /= gamma.sum(axis=1, keepdims=True) + 1e-300
    loglik = np.sum(np.log(c)) + np.sum(m)
    return alpha, beta, gamma, B, c, loglik

def _init(seqs, K, seed=0):
    rng = np.random.default_rng(seed)
    allobs = np.vstack([s["O"] for s in seqs])
    D = allobs.shape[1]
    med = np.nanmedian(allobs, axis=0)
    # severity proxy = mean of standardized obs (already standardized upstream) -> quantile init
    sev = np.nanmean(allobs, axis=1)
    qs = np.nanquantile(sev, np.linspace(0.1, 0.9, K))
    mu = np.tile(med, (K, 1)).astype(float)
    # spread state means along each dim by quantiles of that dim
    for d in range(D):
        col = allobs[:, d]; col = col[~np.isnan(col)]
        if len(col):
            mu[:, d] = np.quantile(col, np.linspace(0.2, 0.8, K))
    var = np.tile(np.nanvar(allobs, axis=0), (K, 1)); var[var < VAR_FLOOR] = 1.0
    pi = np.full(K, 1.0/K)
    A = np.full((K, K), 0.1/(K-1)) + np.eye(K)*0.9
    A /= A.sum(axis=1, keepdims=True)
    return pi, A, mu, var

def fit(seqs, K=4, mode="baseline", n_iter=25, seed=0, verbose=False):
    """seqs: list of dict with 'O' (T x D) and optional 'u' (T,) treatment binary."""
    pi, A, mu, var = _init(seqs, K, seed)
    A_on = A.copy(); A_off = A.copy()
    p_treat = np.linspace(0.2, 0.8, K) if mode == "emission" else None
    D = mu.shape[1]
    prev_ll = -np.inf
    for it in range(n_iter):
        pi_acc = np.zeros(K)
        num_mu = np.zeros((K, D)); den_mu = np.zeros((K, D)); num_var = np.zeros((K, D))
        if mode == "transition":
            xi_off = np.zeros((K, K)); xi_on = np.zeros((K, K))
        else:
            xi_sum = np.zeros((K, K))
        tp_num = np.zeros(K); tp_den = np.zeros(K)
        total_ll = 0.0
        for s in seqs:
            O = s["O"]; T = O.shape[0]
            u = s.get("u") if mode != "baseline" else None
            logB = emission_logB(O, mu, var, u=(u if mode == "emission" else None),
                                 p_treat=(p_treat if mode == "emission" else None))
            if mode == "transition" and u is not None:
                A_t = np.stack([A_on if u[t] > 0.5 else A_off for t in range(T-1)]) if T > 1 \
                      else np.zeros((0, K, K))
            else:
                A_t = np.stack([A]*(T-1)) if T > 1 else np.zeros((0, K, K))
            alpha, beta, gamma, B, c, ll = fwd_bwd(logB, pi, A_t)
            total_ll += ll
            pi_acc += gamma[0]
            mask = ~np.isnan(O)
            # mean AND variance from the SAME posteriors (cached gamma)
            Osq = np.where(mask, O*O, 0.0)
            for k in range(K):
                g = gamma[:, k]
                num_mu[k]  += np.nansum(np.where(mask, O, 0.0) * g[:, None], axis=0)
                den_mu[k]  += (mask * g[:, None]).sum(axis=0)
                num_var[k] += (Osq * g[:, None]).sum(axis=0)   # sum gamma*x^2 (suff. stat)
            for t in range(T-1):
                At = (A_on if (mode == "transition" and u is not None and u[t] > 0.5)
                      else A_off if mode == "transition" else A)
                xi = (alpha[t][:, None] * At * (B[t+1]*beta[t+1])[None, :]) / (c[t+1])
                xi /= xi.sum() + 1e-300
                if mode == "transition" and u is not None:
                    if u[t] > 0.5: xi_on += xi
                    else: xi_off += xi
                else:
                    xi_sum += xi
            if mode == "emission" and u is not None:
                tp_num += (gamma * u[:, None]).sum(axis=0)
                tp_den += gamma.sum(axis=0)
        # M-step
        N = len(seqs)
        pi = pi_acc / N; pi /= pi.sum()
        mu = num_mu / np.maximum(den_mu, 1e-6)
        var = num_var / np.maximum(den_mu, 1e-6) - mu**2   # E[x^2]-E[x]^2
        var[var < VAR_FLOOR] = VAR_FLOOR
        if mode == "transition":
            A_off = xi_off / np.maximum(xi_off.sum(axis=1, keepdims=True), 1e-12)
            A_on  = xi_on  / np.maximum(xi_on.sum(axis=1, keepdims=True), 1e-12)
        else:
            A = xi_sum / np.maximum(xi_sum.sum(axis=1, keepdims=True), 1e-12)
        if mode == "emission":
            p_treat = np.clip(tp_num/np.maximum(tp_den, 1e-6), 1e-4, 1-1e-4)
        if verbose:
            print(f"  iter {it+1}: loglik={total_ll:.1f}")
        if abs(total_ll - prev_ll) < 1e-3*abs(prev_ll+1e-9):
            break
        prev_ll = total_ll
    params = dict(pi=pi, mu=mu, var=var, K=K, mode=mode)
    if mode == "transition": params["A_off"], params["A_on"] = A_off, A_on
    else: params["A"] = A
    if mode == "emission": params["p_treat"] = p_treat
    params["loglik"] = total_ll
    return params

def reorder_by_severity(params, direction):
    """direction: array length D, sign of severity per obs dim. Sort states ascending severity."""
    mu = params["mu"]; sev = (mu * direction[None, :]).sum(axis=1)
    order = np.argsort(sev)
    inv = np.argsort(order)
    p = dict(params)
    p["mu"] = mu[order]; p["var"] = params["var"][order]; p["pi"] = params["pi"][order]
    p["pi"] /= p["pi"].sum()
    if params["mode"] == "transition":
        p["A_off"] = params["A_off"][np.ix_(order, order)]
        p["A_on"]  = params["A_on"][np.ix_(order, order)]
    else:
        p["A"] = params["A"][np.ix_(order, order)]
    if "p_treat" in params: p["p_treat"] = params["p_treat"][order]
    p["_order"] = order
    return p

def posterior_stage(params, O, u=None):
    """Return gamma (T x K) and expected stage per window."""
    mode = params["mode"]; T = O.shape[0]
    logB = emission_logB(O, params["mu"], params["var"],
                         u=(u if mode == "emission" else None),
                         p_treat=(params.get("p_treat") if mode == "emission" else None))
    if mode == "transition" and u is not None:
        A_t = np.stack([params["A_on"] if u[t] > 0.5 else params["A_off"] for t in range(T-1)]) if T>1 else np.zeros((0,params["K"],params["K"]))
    else:
        A = params.get("A", params.get("A_off"))
        A_t = np.stack([A]*(T-1)) if T>1 else np.zeros((0,params["K"],params["K"]))
    _,_,gamma,_,_,ll = fwd_bwd(logB, params["pi"], A_t)
    stages = (gamma * np.arange(params["K"])[None, :]).sum(axis=1)
    return gamma, stages, ll

def obs_pred_loglik(params, seqs):
    """FAIR cross-mode metric: one-step-ahead predictive log-lik of the next
    observation vector, P(O_{t+1} | O_{1:t}, u_{1:t}). Gaussian (lab) part only,
    so emission/transition/baseline are scored on the SAME target."""
    mode = params["mode"]; K = params["K"]; mu = params["mu"]; var = params["var"]
    A_def = params.get("A", params.get("A_off"))
    tot = 0.0; nwin = 0
    for s in seqs:
        O = s["O"]; T = O.shape[0]; u = s.get("u")
        logB = emission_logB(O, mu, var,
                             u=(u if mode == "emission" else None),
                             p_treat=(params.get("p_treat") if mode == "emission" else None))
        m = logB.max(axis=1); B = np.exp(logB - m[:, None])      # full emission for filtering
        gaussB = np.exp(emission_logB(O, mu, var) -              # gaussian-only for prediction
                        emission_logB(O, mu, var).max(axis=1)[:, None])
        gm = emission_logB(O, mu, var).max(axis=1)
        alpha = params["pi"] * B[0]; alpha /= alpha.sum() + 1e-300
        for t in range(T-1):
            if mode == "transition" and u is not None:
                A = params["A_on"] if u[t] > 0.5 else params["A_off"]
            else:
                A = A_def
            pred = alpha @ A
            pdens = (pred * gaussB[t+1]).sum()                   # predictive density (scaled)
            tot += np.log(pdens + 1e-300) + gm[t+1]              # undo gauss scaling
            nwin += 1
            alpha = pred * B[t+1]; alpha /= alpha.sum() + 1e-300
    return tot, nwin, tot/max(nwin, 1)

def heldout_loglik(params, seqs):
    tot = 0.0; nwin = 0
    for s in seqs:
        u = s.get("u")
        _,_,ll = posterior_stage(params, s["O"], u=u)
        tot += ll; nwin += s["O"].shape[0]
    return tot, nwin, tot/max(nwin,1)


## Config

In [2]:
# ============================================================
# CONFIG
# ============================================================
import os, json, warnings, time
import numpy as np
import pandas as pd
from pathlib import Path
warnings.filterwarnings("ignore")

DATA_DIR     = Path(".")
TIMELINE_CSV = DATA_DIR / "cirrhosis_timeline.csv"
COHORT_CSV   = DATA_DIR / "cirrhosis_cohort.csv"
OUT = Path("staging_output"); OUT.mkdir(exist_ok=True)
MIMIC_ROOT   = "/Users/sumiyasayeed/Documents/mimic4/mimiciv"   # for the treatment step

K_STAGES        = 4
OBS_FEATURES    = ["Bilirubin","INR","Albumin","Creatinine","Sodium","MAP"]
N_ITER          = 25
MAX_PATIENTS_FIT = None       # set e.g. 2000 for a fast run; None = all
MIN_WINDOWS     = 2
RANDOM_SEED     = 1
CHUNK           = 200_000

# Treatment classes for the fusion experiment
DRUG_REGEX = {
    "he":       r"lactulose|rifaximin",
    "diuretic": r"furosemide|spironolactone|bumetanide|torsemide",
}
INPUT_ITEMIDS = {                       # icu/inputevents -> d_items
    "vaso":    [221906,221289,221749,222315,221662,221653],
    "albumin": [220864,220862],
}
print("Config loaded. K =", K_STAGES, "| obs =", OBS_FEATURES)


Config loaded. K = 4 | obs = ['Bilirubin', 'INR', 'Albumin', 'Creatinine', 'Sodium', 'MAP']


## Step 1 — Load, standardise, build sequences

In [3]:
# ============================================================
# STEP 1 — LOAD, ROBUST-STANDARDISE, BUILD SEQUENCES (full stay)
# ============================================================
co = pd.read_csv(COHORT_CSV, usecols=["patient_id","is_deceased","decompensated",
                                      "anchor_age","admittime"])
death = dict(zip(co.patient_id, co.is_deceased))
decomp = dict(zip(co.patient_id, co.decompensated))

usecols = ["patient_id","timestamp"] + OBS_FEATURES + ["MELD_Na"]
df = pd.read_csv(TIMELINE_CSV, usecols=usecols, low_memory=False)
if MAX_PATIENTS_FIT:
    keep_pids = pd.Series(df["patient_id"].unique()).sample(
        min(MAX_PATIENTS_FIT, df["patient_id"].nunique()), random_state=RANDOM_SEED)
    df = df[df["patient_id"].isin(set(keep_pids))]

# robust standardisation (median / IQR), saved for reproducibility
STD = {}
for c in OBS_FEATURES:
    v = df[c].dropna().values
    med = float(np.median(v)); iqr = float(np.percentile(v,75)-np.percentile(v,25))
    sd = iqr/1.349 if iqr > 1e-6 else float(np.std(v)); sd = sd if sd > 1e-6 else 1.0
    STD[c] = (med, sd)
    df[c] = np.clip((df[c]-med)/sd, -8, 8)
json.dump(STD, open(OUT/"standardisation.json","w"), indent=2)

df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
seqs = []
for pid, g in df.sort_values("timestamp").groupby("patient_id"):
    g = g.dropna(subset=["timestamp"])
    if len(g) < MIN_WINDOWS: continue
    seqs.append(dict(pid=pid, O=g[OBS_FEATURES].values.astype(float),
                     ts=g["timestamp"].values, meld=g["MELD_Na"].values))
print(f"✅ {len(seqs):,} patient sequences | "
      f"{sum(len(s['O']) for s in seqs):,} windows | D={len(OBS_FEATURES)}")


✅ 7,588 patient sequences | 441,654 windows | D=6


## Step 2 — Fit latent staging HMM + validate ordering

In [4]:
# ============================================================
# STEP 2 — FIT LATENT STAGING HMM + VALIDATE ORDERING
# ============================================================
def apply_order(params, order):
    p = dict(params)
    p["mu"] = params["mu"][order]; p["var"] = params["var"][order]
    p["pi"] = params["pi"][order]; p["pi"] = p["pi"]/p["pi"].sum()
    if params["mode"] == "transition":
        p["A_off"] = params["A_off"][np.ix_(order,order)]
        p["A_on"]  = params["A_on"][np.ix_(order,order)]
    else:
        p["A"] = params["A"][np.ix_(order,order)]
    if "p_treat" in params: p["p_treat"] = params["p_treat"][order]
    return p

def reorder_by_meld(params, seqs):
    """Relabel states by ascending posterior-weighted MELD (clinical severity)."""
    K = params["K"]; num = np.zeros(K); den = np.zeros(K)
    for s in seqs:
        g,_,_ = posterior_stage(params, s["O"])
        meld = s["meld"]; m = ~np.isnan(meld)
        if m.any():
            num += (g[m]*meld[m,None]).sum(0); den += g[m].sum(0)
    sev = num/np.maximum(den,1e-6)
    return apply_order(params, np.argsort(sev)), sev

t0 = time.time()
stage_params = fit(seqs, K=K_STAGES, mode="baseline", n_iter=N_ITER, seed=RANDOM_SEED)
stage_params, _ = reorder_by_meld(stage_params, seqs)
print(f"✅ fit in {time.time()-t0:.1f}s | final loglik={stage_params['loglik']:.0f}")

# patient-level representative stage = peak expected stage over stay
rows = []
for s in seqs:
    g, stages, _ = posterior_stage(stage_params, s["O"])
    rows.append(dict(pid=s["pid"], peak_stage=int(round(stages.max())),
                     mean_stage=float(stages.mean()), mean_meld=np.nanmean(s["meld"])))
P = pd.DataFrame(rows)
P["death"] = P.pid.map(death); P["decomp"] = P.pid.map(decomp)
P.to_csv(OUT/"patient_stages.csv", index=False)

print("\n=== Stage validation (peak stage over stay) ===")
val = P.groupby("peak_stage").agg(n=("pid","size"), mean_MELD=("mean_meld","mean"),
        mortality=("death","mean"), decomp_rate=("decomp","mean")).round(3)
print(val.to_string())
print("\nMonotonic check (should increase 0->3): "
      f"MELD {list(val['mean_MELD'])}, mortality {list(val['mortality'])}")
print("\nBaseline transition matrix (severity-ordered):")
print(np.round(stage_params["A"], 4))
np.save(OUT/"stage_transition_matrix.npy", stage_params["A"])


✅ fit in 44.3s | final loglik=-2956729

=== Stage validation (peak stage over stay) ===
               n  mean_MELD  mortality  decomp_rate
peak_stage                                         
0           3297     11.096      0.361        0.217
1           1704     19.960      0.491        0.479
2           1296     19.287      0.540        0.422
3           1291     29.434      0.696        0.684

Monotonic check (should increase 0->3): MELD [11.096, 19.96, 19.287, 29.434], mortality [0.361, 0.491, 0.54, 0.696]

Baseline transition matrix (severity-ordered):
[[9.943e-01 2.700e-03 2.900e-03 1.000e-04]
 [3.300e-03 9.903e-01 2.100e-03 4.200e-03]
 [5.800e-03 2.700e-03 9.893e-01 2.200e-03]
 [1.000e-04 4.700e-03 1.900e-03 9.933e-01]]


## Step 3 — Treatment extraction (needs MIMIC-IV)

In [5]:
# ============================================================
# STEP 3 — TREATMENT EXTRACTION  (needs MIMIC-IV; required for Step 4)
# Produces a per-window binary "any treatment active" covariate.
# ============================================================
def _csv(d, n):
    for e in (".csv",".csv.gz"):
        if os.path.exists(os.path.join(d, n+e)): return os.path.join(d, n+e)
    return None

def build_treatment_intervals():
    hosp, icu = os.path.join(MIMIC_ROOT,"hosp"), os.path.join(MIMIC_ROOT,"icu")
    co = pd.read_csv(COHORT_CSV, usecols=["subject_id","hadm_id","patient_id","admittime"])
    hadm_set = set(co["hadm_id"].dropna().astype(int))
    hadm_pid = dict(zip(co["hadm_id"], co["patient_id"]))
    intervals = {}   # pid -> list of (start, end)

    pres = _csv(hosp,"prescriptions")
    if pres:
        rx = "|".join(DRUG_REGEX.values())
        for ch in pd.read_csv(pres, usecols=["hadm_id","drug","starttime","stoptime"],
                              chunksize=500_000, low_memory=False):
            ch = ch[ch["hadm_id"].isin(hadm_set)]
            if ch.empty: continue
            ch = ch[ch["drug"].astype(str).str.lower().str.contains(rx, regex=True, na=False)]
            for _, r in ch.iterrows():
                pid = hadm_pid.get(r["hadm_id"]);
                if pid is None: continue
                s = pd.to_datetime(r["starttime"], errors="coerce")
                e = pd.to_datetime(r["stoptime"], errors="coerce")
                if pd.isna(s): continue
                if pd.isna(e): e = s + pd.Timedelta("24h")
                intervals.setdefault(pid, []).append((s, e))
    inp = _csv(icu,"inputevents")
    if inp:
        ids = [i for v in INPUT_ITEMIDS.values() for i in v]
        for ch in pd.read_csv(inp, usecols=["hadm_id","itemid","starttime","endtime"],
                              chunksize=500_000, low_memory=False):
            ch = ch[ch["hadm_id"].isin(hadm_set) & ch["itemid"].isin(ids)]
            if ch.empty: continue
            for _, r in ch.iterrows():
                pid = hadm_pid.get(r["hadm_id"]);
                if pid is None: continue
                s = pd.to_datetime(r["starttime"], errors="coerce")
                e = pd.to_datetime(r["endtime"], errors="coerce")
                if pd.isna(s): continue
                if pd.isna(e): e = s + pd.Timedelta("1h")
                intervals.setdefault(pid, []).append((s, e))
    print(f"✅ treatment intervals for {len(intervals):,} patients")
    return intervals

def attach_treatments(seqs, intervals, win="60min"):
    w = pd.Timedelta(win); n_active = 0
    for s in seqs:
        ts = pd.to_datetime(s["ts"])
        u = np.zeros(len(ts))
        for (st, en) in intervals.get(s["pid"], []):
            active = (ts < (en)) & ((ts + w) > st)
            u[np.asarray(active)] = 1.0
        s["u"] = u; n_active += u.sum()
    print(f"✅ attached treatments | active windows: {int(n_active):,} "
          f"({n_active/sum(len(s['O']) for s in seqs)*100:.1f}%)")

USE_TREATMENTS = os.path.exists(os.path.join(MIMIC_ROOT,"hosp"))
if USE_TREATMENTS:
    intervals = build_treatment_intervals()
    attach_treatments(seqs, intervals)
else:
    print("MIMIC-IV not found — Step 4 will be skipped. Set MIMIC_ROOT to enable.")


✅ treatment intervals for 5,362 patients
✅ attached treatments | active windows: 300,469 (68.0%)


## Step 4 — Transition-fusion experiment

In [6]:
# ============================================================
# STEP 4 — TRANSITION-FUSION EXPERIMENT
# Fair metric: 3-fold CV one-step-ahead predictive log-lik of next labs.
# ============================================================
def cv_compare(seqs, K, n_iter=20, folds=3, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(seqs))
    fold_id = np.array_split(idx, folds)
    out = {m: [] for m in ["baseline","emission","transition"]}
    for f in range(folds):
        te = set(fold_id[f].tolist()); tr = [seqs[i] for i in range(len(seqs)) if i not in te]
        tes = [seqs[i] for i in fold_id[f]]
        for mode in out:
            p = fit(tr, K=K, mode=mode, n_iter=n_iter, seed=seed)
            _,_,per = obs_pred_loglik(p, tes)
            out[mode].append(per)
    return {m: (float(np.mean(v)), float(np.std(v))) for m, v in out.items()}

if "u" in seqs[0]:
    res = cv_compare(seqs, K_STAGES, n_iter=20, folds=3)
    print("=== FAIR one-step predictive log-lik per window (higher = better) ===")
    for m in ["baseline","emission","transition"]:
        mu_, sd_ = res[m]; print(f"  {m:<11} {mu_:.4f} ± {sd_:.4f}")
    best = max(res, key=lambda k: res[k][0])
    print(f"\n  Best-fitting variant: {best}")
    print("  Interpretation: if 'transition' wins, treatment carries information about")
    print("  state DYNAMICS beyond what the labs show; if 'emission' wins, treatment mainly")
    print("  reflects the CURRENT state. (Associational — not causal; confounded by indication.)")

    # Fit a full transition model to inspect treatment-conditioned dynamics
    tp = fit(seqs, K=K_STAGES, mode="transition", n_iter=N_ITER, seed=RANDOM_SEED)
    tp, _ = reorder_by_meld(tp, seqs)
    print("\n  P(progress to a higher stage) by treatment status:")
    for k in range(K_STAGES-1):
        off = tp["A_off"][k, k+1:].sum(); on = tp["A_on"][k, k+1:].sum()
        print(f"    from Stage {k}:  off={off:.3f}  on={on:.3f}")
    np.save(OUT/"A_off.npy", tp["A_off"]); np.save(OUT/"A_on.npy", tp["A_on"])
    json.dump({m:list(v) for m,v in res.items()}, open(OUT/"fusion_cv_results.json","w"), indent=2)
else:
    print("No treatment covariate attached (Step 3 skipped). Run Step 3 with MIMIC-IV to enable.")


=== FAIR one-step predictive log-lik per window (higher = better) ===
  baseline    -6.7741 ± 0.1438
  emission    -6.7445 ± 0.0959
  transition  -6.7737 ± 0.1438

  Best-fitting variant: emission
  Interpretation: if 'transition' wins, treatment carries information about
  state DYNAMICS beyond what the labs show; if 'emission' wins, treatment mainly
  reflects the CURRENT state. (Associational — not causal; confounded by indication.)

  P(progress to a higher stage) by treatment status:
    from Stage 0:  off=0.005  on=0.007
    from Stage 1:  off=0.007  on=0.006
    from Stage 2:  off=0.002  on=0.002


## Step 5 — Visualisation

In [7]:
# ============================================================
# STEP 5 — VISUALISATION
# ============================================================
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
v = P.groupby("peak_stage").agg(mortality=("death","mean"),
        decomp=("decomp","mean"), meld=("mean_meld","mean"), n=("pid","size"))
ax[0].bar(v.index, v["mortality"], color="#C44E52", alpha=.85, label="mortality")
ax[0].bar(v.index, v["decomp"], color="#4C72B0", alpha=.5, label="decompensation")
ax[0].set_xlabel("Latent stage"); ax[0].set_ylabel("rate"); ax[0].set_title("Outcomes by stage")
ax[0].legend(); ax[0].set_xticks(v.index)

ax[1].plot(v.index, v["meld"], "o-", color="#55A868"); ax[1].set_xlabel("Latent stage")
ax[1].set_ylabel("mean MELD-Na"); ax[1].set_title("MELD-Na by stage (external check)")
ax[1].set_xticks(v.index)

im = ax[2].imshow(stage_params["A"], cmap="viridis", vmin=0, vmax=1)
ax[2].set_title("Transition matrix"); ax[2].set_xlabel("to stage"); ax[2].set_ylabel("from stage")
ax[2].set_xticks(range(K_STAGES)); ax[2].set_yticks(range(K_STAGES))
for i in range(K_STAGES):
    for j in range(K_STAGES):
        ax[2].text(j, i, f"{stage_params['A'][i,j]:.2f}", ha="center", va="center",
                   color="w" if stage_params["A"][i,j] < .6 else "k", fontsize=8)
plt.colorbar(im, ax=ax[2], fraction=.046)
plt.tight_layout(); plt.savefig(OUT/"staging_summary.png", dpi=150); plt.close()
print("✅ Saved:", OUT/"staging_summary.png")

if "u" in seqs[0] and os.path.exists(OUT/"A_on.npy"):
    A_off = np.load(OUT/"A_off.npy"); A_on = np.load(OUT/"A_on.npy")
    fig, ax = plt.subplots(1, 2, figsize=(10, 4.2))
    for a, M, t in zip(ax, [A_off, A_on], ["No treatment (A_off)","Treatment active (A_on)"]):
        im = a.imshow(M, cmap="magma", vmin=0, vmax=1); a.set_title(t)
        a.set_xlabel("to stage"); a.set_ylabel("from stage")
        for i in range(K_STAGES):
            for j in range(K_STAGES):
                a.text(j,i,f"{M[i,j]:.2f}",ha="center",va="center",
                       color="w" if M[i,j]<.6 else "k", fontsize=8)
    plt.tight_layout(); plt.savefig(OUT/"transition_fusion.png", dpi=150); plt.close()
    print("✅ Saved:", OUT/"transition_fusion.png")


✅ Saved: staging_output/staging_summary.png
✅ Saved: staging_output/transition_fusion.png
